In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df=pd.read_csv("risk_factors_cervical_cancer.csv")

In [ ]:
df.head()

In [ ]:
cols = df.columns

In [ ]:
cols = ["Age","Smokes","Smokes (years)","Smokes (packs/year)","Hormonal Contraceptives","STDs:HIV","STDs:HPV","Number of sexual partners","First sexual intercourse",'Dx:Cancer', 'Dx:CIN', 'Dx:HPV', 'Dx', 'Hinselmann', 'Schiller',
       'Citology', 'Biopsy']

In [ ]:
df = df[cols]

In [ ]:
df.head(5)

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df['Smokes'] = pd.to_numeric(df['Smokes'],errors="coerce")


In [ ]:
for i in cols:
    df[i] = pd.to_numeric(df[i], errors="coerce")


In [ ]:


binary_cols = ["Hormonal Contraceptives", "STDs:HIV", "STDs:HPV", "IUD", "STDs",
               "STDs:condylomatosis", "STDs:cervical condylomatosis",
               "STDs:vaginal condylomatosis", "STDs:vulvo-perineal condylomatosis",
               "STDs:syphilis", "STDs:pelvic inflammatory disease",
               "STDs:genital herpes", "STDs:molluscum contagiosum",
               "STDs:AIDS", "STDs:Hepatitis B"]

continuous_cols = ["Smokes (years)", "Smokes (packs/year)", "Number of sexual partners",
                    "First sexual intercourse", "Num of pregnancies",
                    "Hormonal Contraceptives (years)", "IUD (years)",
                    "STDs (number)", "STDs: Number of diagnosis"]

for c in binary_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

for c in continuous_cols:
    df[c] = df[c].fillna(df[c].mean())

In [ ]:
df["Smokes"] = df["Smokes"].fillna(df["Smokes"].mode()[0])

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
import seaborn as sns
plt.figure(figsize=(12,4))
sns.jointplot(data=df,x="Age",y="STDs:HPV", hue="Number of sexual partners",palette = "Set1")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

Features and labels


In [ ]:
feature_cols = ['Age', 'Number of sexual partners', 'First sexual intercourse',
       'Num of pregnancies', 'Smokes', 'Smokes (years)', 'Smokes (packs/year)',
       'Hormonal Contraceptives', 'Hormonal Contraceptives (years)', 'IUD',
       'IUD (years)', 'STDs', 'STDs (number)', 'STDs:condylomatosis',
       'STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis',
       'STDs:vulvo-perineal condylomatosis', 'STDs:syphilis',
       'STDs:pelvic inflammatory disease', 'STDs:genital herpes',
       'STDs:molluscum contagiosum', 'STDs:AIDS', 'STDs:HIV',
       'STDs:Hepatitis B', 'STDs:HPV', 'STDs: Number of diagnosis']
X = df[feature_cols]
y= df['Biopsy']
print(y.value_counts())


X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)


In [ ]:
X_train.isnull().sum()

Training logistic Regression Model

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,confusion_matrix

scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled,y_train)

y_prediction = model.predict(X_test_scaled)
print(classification_report(y_test,y_prediction))
print(confusion_matrix(y_test,y_prediction))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_scaled)[:,1]
auc_prob = roc_auc_score(y_test,y_prob)
print(auc_prob)

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X_train_scaled,y_train,cv=5, scoring = "roc_auc")
print(scores)
print(scores.mean())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight="balanced", random_state=42)
rf_scores = cross_val_score(rf_model, X_train_scaled, y_train, cv=5, scoring="roc_auc")
print(rf_scores)
print(rf_scores.mean())

In [ ]:
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))

importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.head(10))

In [ ]:
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

y_pred_rf_lower = (y_prob_rf >= 0.3).astype(int)
print("Threshold = 0.3")
print(classification_report(y_test, y_pred_rf_lower))
print(confusion_matrix(y_test, y_pred_rf_lower))